# OSB Cleaning

In [1]:
import pandas as pd
import re
from datetime import datetime

In [2]:
insolvencies_raw = pd.read_excel("../data/raw/02_monthly_insolvencies.xlsx")

insolvencies_raw.head(5)

,Insolvency Statistics/Statistiques sur l'insolvabilité 1987-2025,Unnamed: 1,Jan/janv 1987,Feb/fév 1987,Mar 1987,Apr/avr 1987,May/mai 1987,Jun/juin 1987,Jul/juil 1987,Aug/août 1987,...,May/mai 2025,Jun/juin 2025,Jul/juill 2025,Aug/août 2025,Sep/Sept 2025,2025-10-01 00:00:00,2025-11-01 00:00:00,2025-12-01 00:00:00,Jan/janv 2026,Feb/fév 2025.1
0,Canada/Canada,Total Insolvencies/Nombre total de dossiers d'...,2765.0,2896.0,2996.0,2720.0,2529.0,2922.0,2487.0,2582.0,...,12395,11883,12661,11631,13072,13091,11904,10989,11775,12715
1,NaN,Total Bankruptcies/Nombre total de faillites,2722.0,2861.0,2928.0,2683.0,2492.0,2880.0,2455.0,2544.0,...,2932,2856,2897,2644,3087,3052,2757,2770,2637,2799
2,NaN,Total Proposals/Nombre total de propositions,43.0,35.0,68.0,37.0,37.0,42.0,32.0,38.0,...,9463,9027,9764,8987,9985,10039,9147,8219,9138,9916
3,NaN,Consumer Insolvencies/Insolvabilité des c...,2060.0,2185.0,2245.0,2071.0,1884.0,2209.0,1912.0,1991.0,...,12004,11464,12285,11303,12668,12662,11548,10652,11408,12307
4,NaN,Consumer Bankruptcies/Faillites des conso...,2037.0,2162.0,2202.0,2046.0,1866.0,2180.0,1890.0,1962.0,...,2631,2529,2611,2383,2793,2724,2484,2510,2349,2479


In [3]:
region_col = insolvencies_raw.columns[0]
insolvencies = insolvencies_raw.copy()
insolvencies[region_col] = insolvencies[region_col].ffill()


def parse_month_column(col):
    if isinstance(col, (pd.Timestamp, datetime)):
        return pd.Timestamp(col).normalize()
    s = str(col).strip()
    s = re.sub(r"\.\d+$", "", s)
    if "/" in s:
        eng_month = s.split("/")[0].strip()
        year = s.split()[-1]
        return pd.to_datetime(f"{eng_month} {year}", format="%b %Y")
    return pd.to_datetime(s)


_meta = list(insolvencies.columns[:2])
_period_headers = [parse_month_column(c) for c in insolvencies.columns[2:]]
insolvencies.columns = _meta + _period_headers

_meta_block = insolvencies.iloc[:, :2].copy()
_value_block = insolvencies.iloc[:, 2:].apply(pd.to_numeric, errors="coerce").astype("float64")
insolvencies = pd.concat([_meta_block, _value_block], axis=1)

In [4]:
_region = insolvencies.columns[0]
_metric = insolvencies.columns[1]
_period = insolvencies.columns[2:]

_ts_to_cols = {}
_ts_order = []
for c in _period:
    if c not in _ts_to_cols:
        _ts_to_cols[c] = []
        _ts_order.append(c)
    _ts_to_cols[c].append(c)

_parts = [insolvencies[[_region, _metric]].copy()]
for ts in _ts_order:
    cols = _ts_to_cols[ts]
    if len(cols) == 1:
        _parts.append(insolvencies[cols[0]].rename(ts))
    else:
        # duplicate month in source (e.g. two Feb 2025): combine by row-wise mean
        _parts.append(insolvencies[cols].mean(axis=1).rename(ts))

_wide_for_melt = pd.concat(_parts, axis=1)

METRIC_ORDER = [
    "Total Insolvencies",
    "Total Bankruptcies",
    "Total Proposals",
    "Consumer Insolvencies",
    "Consumer Bankruptcies",
    "Consumer Proposals",
    "Business Insolvencies",
    "Business Bankruptcies",
    "Business Proposals",
    "Corporate Insolvencies",
    "Corporate Bankruptcies",
    "Corporate Proposals",
    "Individual Business Insolvencies",
    "Individual Business Bankruptcies",
    "Individual Business Proposals",
]

_long = _wide_for_melt.melt(
    id_vars=[_region, _metric], var_name="date", value_name="value"
)
_long["location"] = _long[_region].map(lambda s: str(s).split("/")[0].strip())
_long["metric"] = _long[_metric].map(lambda s: str(s).split("/")[0].strip())
_long = _long.drop(columns=[_region, _metric])

_wide = (
    _long.pivot_table(
        index=["date", "location"], columns="metric", values="value", aggfunc="mean"
    )
    .reset_index()
)
_wide.columns.name = None

assert set(METRIC_ORDER).issubset(_wide.columns), (
    f"missing metrics: {set(METRIC_ORDER) - set(_wide.columns)}"
)

insolvencies_by_month_location = _wide.reindex(
    columns=["date", "location"] + METRIC_ORDER
)
insolvencies_by_month_location["date"] = pd.to_datetime(
    insolvencies_by_month_location["date"]
).dt.strftime("%Y-%m-%d")
insolvencies_by_month_location = insolvencies_by_month_location.sort_values(
    ["date", "location"]
).reset_index(drop=True)

In [5]:
SUMMED_COLS = [
    "Total Insolvencies",
    "Total Bankruptcies",
    "Total Proposals",
    "Consumer Insolvencies",
    "Business Insolvencies",
    "Business Bankruptcies",
    "Business Proposals",
    "Corporate Insolvencies",
    "Individual Business Insolvencies",
]

insolvencies_clean = insolvencies_by_month_location.drop(columns=SUMMED_COLS).copy()
insolvencies_clean.head()

,date,location,Consumer Bankruptcies,Consumer Proposals,Corporate Bankruptcies,Corporate Proposals,Individual Business Bankruptcies,Individual Business Proposals
0,1987-01-01,Alberta,224.0,1.0,11.0,0.0,95.0,2.0
1,1987-01-01,British Columbia,193.0,2.0,29.0,1.0,87.0,0.0
2,1987-01-01,Canada,2037.0,23.0,176.0,14.0,509.0,6.0
3,1987-01-01,Manitoba,96.0,1.0,2.0,0.0,24.0,0.0
4,1987-01-01,New Brunswick,29.0,0.0,1.0,0.0,7.0,1.0


In [6]:
LOCATION_MAP = {
    "British Columbia": "BC",
    "Alberta": "AB",
    "Saskatchewan": "SK",
    "Manitoba": "MB",
    "Ontario": "ON",
    "Quebec": "QC",
    "Québec": "QC",
    "New Brunswick": "NB",
    "Nova Scotia": "NS",
    "Prince Edward Island": "PE",
    "Newfoundland and Labrador": "NL",
    "Newfoundland & Labrador": "NL",
    "Newfoundland": "NL",
    "Yukon": "YT",
    "Yukon Territory": "YT",
    "Northwest Territories": "NT",
    "Northwest Territory": "NT",
    "Nunavut": "NU",
    "Canada": "CAN",
}

_loc_raw = insolvencies_clean["location"].str.strip()
insolvencies_clean["location"] = _loc_raw.map(LOCATION_MAP)

# Sanity check — surface any unmapped values (NaN means no match found)
unmapped_mask = insolvencies_clean["location"].isna()
if unmapped_mask.any():
    print("Unmapped locations:", _loc_raw[unmapped_mask].unique())
else:
    print("All locations mapped successfully.")

All locations mapped successfully.


In [7]:
VALUE_COLS = [c for c in insolvencies_clean.columns if c not in ("date", "location")]

# Convert to nullable integer (handles NaN without float coercion)
insolvencies_clean[VALUE_COLS] = (
    insolvencies_clean[VALUE_COLS]
    .round(0)
    .astype(pd.Int64Dtype())
)

# --- EDA: nulls ---
print("=== Shape ===")
print(insolvencies_clean.shape)

print("\n=== Dtypes ===")
print(insolvencies_clean.dtypes)

print("\n=== Null counts per column ===")
null_counts = insolvencies_clean.isnull().sum()
print(null_counts)

print("\n=== % Null per column ===")
print((null_counts / len(insolvencies_clean) * 100).round(2))

print("\n=== Rows with any null ===")
null_rows = insolvencies_clean[insolvencies_clean.isnull().any(axis=1)]
print(f"{len(null_rows)} rows have at least one null value")
print(null_rows["location"].value_counts())

=== Shape ===
(6419, 8)

=== Dtypes ===
date                                  str
location                              str
Consumer Bankruptcies               Int64
Consumer Proposals                  Int64
Corporate Bankruptcies              Int64
Corporate Proposals                 Int64
Individual Business Bankruptcies    Int64
Individual Business Proposals       Int64
dtype: object

=== Null counts per column ===
date                                0
location                            0
Consumer Bankruptcies               0
Consumer Proposals                  0
Corporate Bankruptcies              0
Corporate Proposals                 0
Individual Business Bankruptcies    0
Individual Business Proposals       0
dtype: int64

=== % Null per column ===
date                                0.0
location                            0.0
Consumer Bankruptcies               0.0
Consumer Proposals                  0.0
Corporate Bankruptcies              0.0
Corporate Proposals               

In [8]:
insolvencies_clean

,date,location,Consumer Bankruptcies,Consumer Proposals,Corporate Bankruptcies,Corporate Proposals,Individual Business Bankruptcies,Individual Business Proposals
0,1987-01-01,AB,224,1,11,0,95,2
1,1987-01-01,BC,193,2,29,1,87,0
2,1987-01-01,CAN,2037,23,176,14,509,6
3,1987-01-01,MB,96,1,2,0,24,0
4,1987-01-01,NB,29,0,1,0,7,1
...,...,...,...,...,...,...,...,...
6414,2026-01-01,ON,764,3405,72,8,25,12
6415,2026-01-01,PE,11,37,0,0,0,0
6416,2026-01-01,QC,861,2176,138,32,18,9
6417,2026-01-01,SK,60,216,0,0,0,1


In [9]:
_q = insolvencies_clean.copy()
_q["date"] = pd.to_datetime(_q["date"])
_q["quarter"] = _q["date"].dt.year.astype(str) + "_Q" + _q["date"].dt.quarter.astype(str)

VALUE_COLS = [c for c in _q.columns if c not in ("date", "quarter", "location")]

insolvencies_by_quarter = (
    _q.groupby(["quarter", "location"], sort=True)[VALUE_COLS]
    .sum()
    .reset_index()
)
insolvencies_by_quarter[VALUE_COLS] = insolvencies_by_quarter[VALUE_COLS].astype(pd.Int64Dtype())

insolvencies_by_quarter

,quarter,location,Consumer Bankruptcies,Consumer Proposals,Corporate Bankruptcies,Corporate Proposals,Individual Business Bankruptcies,Individual Business Proposals
0,1987_Q1,AB,681,2,34,1,250,2
1,1987_Q1,BC,702,4,76,9,292,4
2,1987_Q1,CAN,6401,89,525,40,1585,17
3,1987_Q1,MB,277,3,6,1,68,1
4,1987_Q1,NB,71,2,3,0,17,1
...,...,...,...,...,...,...,...,...
2144,2026_Q1,ON,764,3405,72,8,25,12
2145,2026_Q1,PE,11,37,0,0,0,0
2146,2026_Q1,QC,861,2176,138,32,18,9
2147,2026_Q1,SK,60,216,0,0,0,1


In [ ]:
from pathlib import Path

out_path = Path("../data/processed/02_monthly_insolvencies_processed.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)
insolvencies_clean.to_csv(out_path, index=False)
print(f"Wrote {len(insolvencies_clean):,} rows to {out_path}")